In [1]:
"""

# 🎵 Music Recommender System

A content-based music recommender system built on 114K Spotify tracks.

## How it works
- Each song is represented as a **128-dimensional feature vector**
- Audio features (energy, danceability, valence, tempo...) are normalized using MinMaxScaler
- Genres are one-hot encoded
- **Cosine similarity** is computed on-demand to avoid storing 6.6 billion entries

## Usage
```python
recommend("Believer", artist="Imagine Dragons")
recommend("Shape of You", n=10)
```

"""

'\n\n# 🎵 Music Recommender System\n\nA content-based music recommender system built on 114K Spotify tracks.\n\n## How it works\n- Each song is represented as a **128-dimensional feature vector**\n- Audio features (energy, danceability, valence, tempo...) are normalized using MinMaxScaler\n- Genres are one-hot encoded\n- **Cosine similarity** is computed on-demand to avoid storing 6.6 billion entries\n\n## Usage\n```python\nrecommend("Believer", artist="Imagine Dragons")\nrecommend("Shape of You", n=10)\n```\n\n'

In [2]:
# Imports 

import numpy as np
import pandas as pd

In [3]:
# Step 1: Load Data
music = pd.read_csv('dataset.csv')
music.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
music.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [5]:
# Step 2: Data Cleaning

# Remove duplicates
music = music.drop_duplicates(subset=['track_name', 'artists'], keep='first', ignore_index=True)

# Drop useless columns
music = music.drop(columns=['Unnamed: 0', 'track_id', 'album_name'])

# Drop rows with missing track name or artist
music = music.dropna(subset=['track_name', 'artists'])
music = music.reset_index(drop=True)

print("Clean dataset shape:", music.shape)

Clean dataset shape: (81343, 18)


In [6]:
# Step 3: Feature Engineering

from sklearn.preprocessing import MinMaxScaler

# Convert explicit boolean to 0/1
music['explicit'] = music['explicit'].astype(int)

# Normalize all numeric columns
feature_cols = music.select_dtypes(include=['float64', 'int64']).columns.tolist()
feature_cols = [col for col in feature_cols if col != 'explicit']

scaler = MinMaxScaler()
music[feature_cols] = scaler.fit_transform(music[feature_cols])

# One-hot encode genre
genre_dummies = pd.get_dummies(music['track_genre'], prefix='genre')

# Build final feature matrix
feature_matrix = pd.concat([
    music[feature_cols],
    music[['explicit']],
    genre_dummies
], axis=1)

print("Feature matrix shape:", feature_matrix.shape)

Feature matrix shape: (81343, 128)


In [7]:
# Step 4: Recommendation Engine

from sklearn.metrics.pairwise import cosine_similarity

def recommend(song, artist=None, n=5):

    # Convert input to lowercase
    song = song.lower()

    # Search for the song
    match = music[music['track_name'].str.lower() == song]

    # If artist is provided, filter directly
    if artist:
        match = match[
            match['artists'].str.lower().str.contains(artist.lower())
        ]

    # If multiple songs exist and no artist is given, let user pick
    elif len(match) > 1:
        print(f"\nMultiple songs named '{song.title()}' found:\n")
        for i, (_, row) in enumerate(match.iterrows(), start=1):
            print(f"{i}. {row['track_name']} - {row['artists']} ({row['track_genre']})")
        choice = int(input("\nEnter song number: "))
        match = match.iloc[[choice - 1]]

    # If song is not found
    if match.empty:
        print("Sorry! This song was not found.\n")
        print("Did you mean one of these?\n")
        suggestions = music[
            music['track_name'].str.lower().str.contains(song, na=False)
        ]
        if suggestions.empty:
            print("No similar songs found.")
        else:
            for s in suggestions['track_name'].unique()[:10]:
                print(s)
        return

    # Song found
    song_idx = match.index[0]
    song_info = music.iloc[song_idx]

    # Get feature vector
    song_vector = feature_matrix.iloc[song_idx].values.reshape(1, -1)

    # Compute similarity
    similarity = cosine_similarity(song_vector, feature_matrix)

    # Pair indices with similarity scores
    distances = sorted(
        list(enumerate(similarity[0])),
        key=lambda x: x[1],
        reverse=True
    )

    # ---------------- Selected Song ----------------

    print("=" * 60)
    print("SELECTED SONG")
    print("=" * 60)
    print(f"Track        : {song_info['track_name']}")
    print(f"Artist       : {song_info['artists']}")
    print(f"Genre        : {song_info['track_genre']}")
    print(f"Popularity   : {int(song_info['popularity'] * 100)}%")
    print(f"Explicit     : {'Yes' if song_info['explicit'] else 'No'}")
    print(f"Danceability : {song_info['danceability']:.2f}")
    print(f"Energy       : {song_info['energy']:.2f}")
    print(f"Tempo        : {song_info['tempo']:.2f}")
    print("=" * 60)

    # ---------------- Recommendations ----------------

    print(f"\nTop {n} Similar Songs\n")

    count = 0
    for idx, score in distances:

        if idx == song_idx:
            continue

        rec = music.iloc[idx]

        print("-" * 60)
        print(f"Track        : {rec['track_name']}")
        print(f"Artist       : {rec['artists']}")
        print(f"Genre        : {rec['track_genre']}")
        print(f"Popularity   : {int(rec['popularity'] * 100)}%")
        print(f"Similarity   : {score * 100:.2f}%")

        count += 1
        if count == n:
            break

    print("-" * 60)

In [8]:
# Test
recommend("Believer", artist="Imagine Dragons")

SELECTED SONG
Track        : Believer
Artist       : Imagine Dragons
Genre        : rock
Popularity   : 88%
Explicit     : No
Danceability : 0.79
Energy       : 0.78
Tempo        : 0.51

Top 5 Similar Songs

------------------------------------------------------------
Track        : Whatever It Takes
Artist       : Imagine Dragons
Genre        : rock
Popularity   : 81%
Similarity   : 99.62%
------------------------------------------------------------
Track        : Glad You Came
Artist       : The Wanted
Genre        : rock
Popularity   : 79%
Similarity   : 98.95%
------------------------------------------------------------
Track        : House of Memories
Artist       : Panic! At The Disco
Genre        : rock
Popularity   : 85%
Similarity   : 98.94%
------------------------------------------------------------
Track        : Electric Love
Artist       : BØRNS
Genre        : rock
Popularity   : 82%
Similarity   : 98.35%
------------------------------------------------------------
Track 

In [9]:
"""

# Scale dynamically based on the column's actual min and max values
"""
#Scaling means making values in comparable range so did this  => (val - min) / (max - min) this is the formula

"""
music['loudness'] = (music['loudness'] - music['loudness'].min()) / (music['loudness'].max() - music['loudness'].min())
music['popularity'] = (music['popularity'] - music['popularity'].min()) / (music['popularity'].max() - music['popularity'].min())
music['duration_ms'] = (music['duration_ms'] - music['duration_ms'].min()) / (music['duration_ms'].max() - music['duration_ms'].min())

music['key'] = (music['key'] - music['key'].min()) / (music['key'].max() - music['key'].min())
music['tempo'] = (music['tempo'] - music['tempo'].min()) / (music['tempo'].max() - music['tempo'].min())
music['time_signature'] = (music['time_signature'] - music['time_signature'].min()) / (music['time_signature'].max() - music['time_signature'].min())



# Convert explicit boolean to 1 and 0
music['explicit'] = music['explicit'].astype(int)

"""


"\nmusic['loudness'] = (music['loudness'] - music['loudness'].min()) / (music['loudness'].max() - music['loudness'].min())\nmusic['popularity'] = (music['popularity'] - music['popularity'].min()) / (music['popularity'].max() - music['popularity'].min())\nmusic['duration_ms'] = (music['duration_ms'] - music['duration_ms'].min()) / (music['duration_ms'].max() - music['duration_ms'].min())\n\nmusic['key'] = (music['key'] - music['key'].min()) / (music['key'].max() - music['key'].min())\nmusic['tempo'] = (music['tempo'] - music['tempo'].min()) / (music['tempo'].max() - music['tempo'].min())\nmusic['time_signature'] = (music['time_signature'] - music['time_signature'].min()) / (music['time_signature'].max() - music['time_signature'].min())\n\n\n\n# Convert explicit boolean to 1 and 0\nmusic['explicit'] = music['explicit'].astype(int)\n\n"

In [10]:
"""

# failed bcz it takes lot of space(81344 × 81344) =>  6.6 billion numbers

from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(feature_matrix)

similarity.shape

"""

'\n\n# failed bcz it takes lot of space(81344 × 81344) =>  6.6 billion numbers\n\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nsimilarity = cosine_similarity(feature_matrix)\n\nsimilarity.shape\n\n'